# Notebook to calculate the ACF for all gene constructs.
____

### Loading libraries and source code
____

In [ ]:
import sys; from pathlib import Path; current_dir = Path().resolve()
src_dir = next(parent / 'src' for parent in Path().absolute().parents if (parent / 'src').is_dir())
sys.path.extend([str(src_dir), str(src_dir / 'pipelines')])
main_dir = Path(src_dir.parents[0])
from imports import * 
import joblib

### Loading main folders and data
_____

In [ ]:
#path_main_folder = Path ('/Users/nzlab-la/Library/CloudStorage/OneDrive-TheUniversityofColoradoDenver/General - Zhao (NZ) Lab/Zhao lab shared folder/Our papers/Anti-Utag-frankenbody paper/Raw_Data_ACF_HT/ACF_data')
path_main_folder = Path ('/Volumes/LaCie/UTag_paper_data/ACF')
results_folder = path_main_folder.joinpath( 'results_ACF')
if not results_folder.exists():
    results_folder.mkdir()

### Parameters 
____

In [ ]:
step_size_in_sec = 5
start_lag = 0
remove_outliers = True
selected_field = 'spot_int_ch_0'  # spot_int_ch_0, psf_amplitude_ch_0, snr_ch_0,
min_percentage_data_in_trajectory = 0.3
max_missing_frames = 5
downsample = False
downsampling_factor = 2
control_spots_mode = False
list_datasets = [ 'utag', 'utag_c_free' , 'suntag', 'alfatag']
y_axes_min_max_list_values =  [-0.01, 0.1]
x_axes_min_max_list_values = [-0.2, 25.5]
max_lag = 100
if control_spots_mode:
    y_axes_min_max_list_values = None # [-20, 20]
MAD_THRESHOLD_FACTOR = 4 # 4 was working well for the control spots. 
multi_tau=False
multi_tau_raw_points = 50
multi_tau_bins_per_stage = 40
min_snr=0.5

#### Main Functions
____

In [ ]:
def dataset_selection(dataset,control_spots_mode=False,downsample=downsample):
    if control_spots_mode == True:
        dataframe_prefix='random_location_spots_'
    else:
        dataframe_prefix='tracking_'
    if dataset == 'suntag':
        folder_with_files = path_main_folder.joinpath('SunTag_ACF')
        plot_name = 'suntag'  
    elif dataset == 'utag':
        plot_name = 'utag'
        folder_with_files = path_main_folder.joinpath('UTag_ACF')
    elif dataset == 'alfatag':
        plot_name = 'alfatag'
        folder_with_files = path_main_folder.joinpath('AlfaTag_ACF')
    elif dataset == 'utag_c_free':
        plot_name = 'utag_c_free'
        folder_with_files = path_main_folder.joinpath('UTag_CF_ACF')
    if control_spots_mode == True:
        plot_name = plot_name + '_control_spots'
    if downsample == True:
        plot_name = plot_name + '_downsampled'
    return folder_with_files, plot_name, dataframe_prefix

In [ ]:
def extract_intensity_and_shift_data(dataframe, selected_field='spot_int_ch_0', 
                                   min_percentage_data_in_trajectory=0.4, 
                                   padd_with_nans=True, maximum_columns=360, 
                                   min_snr=1, max_missing_frames=5):
    """
    Extract intensity data with proper SNR filtering applied before trajectory shifting.
    """
    # Extract raw arrays
    intensity_array = mi.Utilities().df_trajectories_to_array(
        dataframe=dataframe, selected_field=selected_field, fill_value='nans')
    snr_array = mi.Utilities().df_trajectories_to_array(
        dataframe=dataframe, selected_field='snr_ch_' + selected_field[-1], fill_value='nans')    
    mean_snr_array = np.nanmean(snr_array, axis=1)    
    snr_mask = mean_snr_array >= min_snr
    intensity_array_filtered = intensity_array[snr_mask, :]    
    intensity_array_shifted = mi.Utilities().shift_trajectories(
        intensity_array_filtered,
        min_percentage_data_in_trajectory=min_percentage_data_in_trajectory,
        max_missing_frames=max_missing_frames)
    number_of_columns = intensity_array_shifted.shape[1]
    if padd_with_nans:        
        if number_of_columns < maximum_columns:
            intensity_array_shifted = np.pad(intensity_array_shifted, 
                                           ((0,0), (0, maximum_columns-number_of_columns)), 
                                           mode='constant', constant_values=np.nan)
    if number_of_columns > maximum_columns:
        intensity_array_shifted = intensity_array_shifted[:, :maximum_columns]
        
    return intensity_array_shifted

In [ ]:
def load_tracking_data(root_folder: Path, selected_field: str,min_percentage_data_in_trajectory=0.3, dataframe_prefix='tracking_',min_snr=1):
    """
    Load and process all tracking CSV files found in subfolders whose names include 'results_'.
    Extracts intensity and shift data from each file and concatenates the results along axis=0.
    """
    # Get subfolders with 'results_' in the name.
    subfolders = [folder for folder in root_folder.iterdir() if folder.is_dir() and 'results_' in folder.name]
    tracking_files = [file 
                      for folder in subfolders 
                      for file in folder.iterdir() 
                      if dataframe_prefix in file.name and file.suffix.lower() == '.csv' ]
    intensity_arrays = []
    number_of_cells = 0
    for tracking_file in tracking_files:
        df = pd.read_csv(tracking_file, encoding='latin-1')
        try:
            intensity_array = extract_intensity_and_shift_data(df, selected_field=selected_field,min_percentage_data_in_trajectory=min_percentage_data_in_trajectory,min_snr=min_snr)
            intensity_arrays.append(intensity_array)
            number_of_cells += 1
        except:
            print(f'Error processing {tracking_file}')
    concatenated_intensity_arrays = np.concatenate(intensity_arrays, axis=0)
    total_number_of_spots = np.shape(concatenated_intensity_arrays)[0]
    return concatenated_intensity_arrays, number_of_cells, total_number_of_spots


In [ ]:
def plot_data_matrix(array_int_all_days, results_folder, plot_name):
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    data = array_int_all_days
    ax.imshow(data, aspect='auto', cmap='viridis', interpolation='none')
    cbar = plt.colorbar(ax.imshow(data, aspect='auto', cmap='viridis', interpolation='none'))
    cbar.set_label('RNA spot intensity')
    cbar.ax.invert_yaxis()
    ax.set_xlabel('Shifted Frames (5 sec steps)')
    ax.set_ylabel('Spots')
    ax.set_title('Spot intensity')
    plt.savefig(results_folder.joinpath( 'raw_data_'+plot_name+'.png'), dpi=300)
    plt.show()

In [ ]:
def plot_correlation_comparison(mean_correlation, std_correlation, lags,mean_correlation_ssa=None, std_correlation_ssa=None, lags_ssa=None,  
                                plot_name='temp', start_lag=1, y_axes_min_max_list_values = None,x_axes_min_max_list_values=None,show_legend = True):

    # Update rcParams to set a white background and Arial fonts.
    plt.rcParams.update({
        'figure.facecolor': 'white',    # figure background is white
        'axes.facecolor': 'white',      # axes (plot area) background is white
        'axes.edgecolor': 'black',      # axis spines (box) will be black
        'axes.linewidth': 1.5,          # thicker border lines for a clear box
        'font.family': 'sans-serif',
        'font.sans-serif': 'Arial',
        'axes.labelsize': 14,           # axis labels: minimum 12
        'axes.titlesize': 14,           # title font size: 16
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'axes.labelcolor': 'black',
        'text.color': 'black',
        'xtick.color': 'black',
        'ytick.color': 'black',
        'axes.edgecolor': 'black',
    })
    # Create the figure and axes.
    fig, ax = plt.subplots(figsize=(5, 3), facecolor='white')
    lags = lags / 60  # convert seconds to minutes
    maxlag = len(lags)
    ax.plot(lags[start_lag:maxlag], mean_correlation[start_lag:maxlag],
            'o-', color='dimgray', linewidth=1, label='Experimental', alpha=0.5)
    ax.fill_between(lags[start_lag:maxlag],
                    mean_correlation[start_lag:maxlag] - std_correlation[start_lag:maxlag],
                    mean_correlation[start_lag:maxlag] + std_correlation[start_lag:maxlag],
                    color='dimgray', alpha=0.5)
    if mean_correlation_ssa is not None:
        # Plot the simulation correlation (red).
        ax.plot(lags_ssa[start_lag:maxlag], mean_correlation_ssa[start_lag:maxlag],
                'o-', color='goldenrod', linewidth=1, label='Simulation', alpha=0.5)
        ax.fill_between(lags_ssa[start_lag:maxlag],
                        mean_correlation_ssa[start_lag:maxlag] - std_correlation_ssa[start_lag:maxlag],
                        mean_correlation_ssa[start_lag:maxlag] + std_correlation_ssa[start_lag:maxlag],
                        color='goldenrod', alpha=0.5)
    # Add a legend.
    if show_legend:
        ax.legend(loc='upper right')
    # Set axis limits.
    if x_axes_min_max_list_values is not None:
        ax.set_xlim(x_axes_min_max_list_values[0],x_axes_min_max_list_values[1])

    if y_axes_min_max_list_values is not None:
        ax.set_ylim(y_axes_min_max_list_values[0],y_axes_min_max_list_values[1])
    # Set axis labels and title.
    ax.set_xlabel(r'$\tau (s)$', fontsize=14)
    ax.set_ylabel(r'$G(\tau)$', fontsize=14)
    #ax.set_title(plot_name)
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    # save the figure and ensure the layout is tight
    plt.tight_layout()
    # save the figure 
    plt.savefig(results_folder.joinpath('ACF_'+plot_name+'.png'), dpi=600)
    plt.show()

In [ ]:
def plot_all_images_in_a_folder(folder_with_files, plot_name, results_folder):
    list_images = []
    list_image_names = []
    subfolders = [folder for folder in folder_with_files.iterdir() if folder.is_dir()]
    subfolders = [folder for folder in subfolders if 'results_' in folder.name]
    # sort the subfolders by name and the first string on it. 
    subfolders = sorted(subfolders, key=lambda x: x.name.split('_')[1])
    for subfolder in subfolders:
        #print(f'Processing subfolder: {subfolder}')
        files = [f for f in subfolder.iterdir() if f.is_file() and 'correlation' in f.name and f.suffix == '.png']
        # save name removing "correlation_image" from the name
        list_image_names.append(subfolder.name.replace('results_', ''))
        image = [Image.open(f) for f in files][0]
        image = np.array(image)
        list_images.append(image)
    num_cols = 5
    num_rows = int(np.ceil(len(list_images) / num_cols))
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(12, 12))
    # plot the images in the grid
    for i, image in enumerate(list_images):
        ax = axs[i // num_cols, i % num_cols]
        ax.imshow(image)
        ax.axis('off')
        ax.set_title(list_image_names[i], fontsize=6)
    for j in range(i + 1, num_rows * num_cols):
        ax = axs[j // num_cols, j % num_cols]
        ax.axis('off')
    # thigh layout
    plt.tight_layout()
    fig.suptitle(f'{plot_name} ACF ', fontsize=20)

In [ ]:
def plot_trajectory_timecourses(array_int_all_days, num_trajectories_plot=20, 
                               figsize=(10, 6), alpha=0.6, linewidth=1.2, 
                               time_interval_seconds=5, 
                               save_path=None, y_label="Intensity (a.u.)", x_label="Time (min)",
                               intensity_matching=True, intensity_tolerance=0.2):
    """
    Plot time courses for selected trajectories, sorted by quality and intensity similarity.
    
    Parameters:
    -----------
    array_int_all_days : numpy.ndarray
        2D array where rows are trajectories and columns are time points
    num_trajectories_plot : int
        Number of trajectories to plot
    figsize : tuple
        Figure size (width, height) in inches
    alpha : float
        Transparency of trajectory lines
    linewidth : float
        Width of trajectory lines
    time_interval_seconds : int
        Time interval between frames in seconds
    save_path : str or Path
        Path to save the plot (optional)
    y_label, x_label : str
        Axis labels
    intensity_matching : bool
        If True, select trajectories with similar average intensities
    intensity_tolerance : float
        Acceptable deviation from median intensity (as fraction of median)
    
    Returns:
    --------
    fig, ax : matplotlib objects
    selected_indices : numpy.ndarray
        Indices of selected trajectories
    """
    n_trajectories, n_timepoints = array_int_all_days.shape
    num_trajectories_plot = min(num_trajectories_plot, n_trajectories)
    
    # Calculate quality scores and intensity metrics for each trajectory
    quality_scores = np.zeros(n_trajectories)
    mean_intensities = np.zeros(n_trajectories)
    
    for i in range(n_trajectories):
        trajectory = array_int_all_days[i, :]
        non_nan_count = np.sum(~np.isnan(trajectory))
        total_length = len(trajectory)
        completeness = non_nan_count / total_length
        valid_mask = ~np.isnan(trajectory)
        consecutive_lengths = []
        current_length = 0
        for is_valid in valid_mask:
            if is_valid:
                current_length += 1
            else:
                if current_length > 0:
                    consecutive_lengths.append(current_length)
                current_length = 0
        if current_length > 0:
            consecutive_lengths.append(current_length)
        max_consecutive = max(consecutive_lengths) if consecutive_lengths else 0
        consecutive_score = max_consecutive / total_length
        quality_scores[i] = 0.5 * completeness + 0.5 * consecutive_score
        
        # Calculate mean intensity for valid data points
        valid_intensity = trajectory[~np.isnan(trajectory)]
        if len(valid_intensity) > 0:
            mean_intensities[i] = np.mean(valid_intensity)
        else:
            mean_intensities[i] = np.nan
    
    # Select trajectories based on intensity matching or quality only
    if intensity_matching:
        # Remove trajectories with NaN intensities
        valid_intensity_mask = ~np.isnan(mean_intensities)
        
        if np.sum(valid_intensity_mask) == 0:
            print("No trajectories with valid intensities found!")
            return None, None, None
        
        # Calculate target intensity (median of all valid trajectories)
        target_intensity = np.median(mean_intensities[valid_intensity_mask])
        intensity_range = intensity_tolerance * target_intensity
        
        # Filter trajectories within intensity range
        intensity_mask = (np.abs(mean_intensities - target_intensity) <= intensity_range) & valid_intensity_mask
        candidate_indices = np.where(intensity_mask)[0]
        
        if len(candidate_indices) < num_trajectories_plot:
            # Gradually increase tolerance until we have enough trajectories
            expanded_tolerance = intensity_tolerance
            while len(candidate_indices) < num_trajectories_plot and expanded_tolerance < 1.0:
                expanded_tolerance += 0.1
                intensity_range = expanded_tolerance * target_intensity
                intensity_mask = (np.abs(mean_intensities - target_intensity) <= intensity_range) & valid_intensity_mask
                candidate_indices = np.where(intensity_mask)[0]
        
        # Sort candidates by quality and select top ones
        candidate_qualities = quality_scores[candidate_indices]
        sorted_candidate_indices = candidate_indices[np.argsort(candidate_qualities)[::-1]]
        selected_indices = sorted_candidate_indices[:num_trajectories_plot]
        
    else:
        # Sort trajectories by quality (highest first)
        sorted_indices = np.argsort(quality_scores)[::-1]
        selected_indices = sorted_indices[:num_trajectories_plot]
    
    # Create time array
    time_array = np.arange(n_timepoints) * time_interval_seconds / 60
    
    # Set up colors
    colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan',
              'navy', 'teal', 'magenta', 'gold', 'lime', 'coral', 'darkred', 'darkgreen', 'darkblue', 'darkorange']
    
    # Create the plot
    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    ax.set_facecolor('white')
    
    # Plot selected trajectories
    for i, traj_idx in enumerate(selected_indices):
        trajectory = array_int_all_days[traj_idx, :]
        color = colors[i % len(colors)]  # Cycle through colors if more trajectories than colors
        ax.plot(time_array, trajectory, color=color, alpha=alpha, linewidth=linewidth)
    
    # Customize axes
    ax.set_xlabel(x_label, fontsize=16, color='black')
    ax.set_ylabel(y_label, fontsize=16, color='black')
    
    # Customize spines
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.5)
    
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    ax.tick_params(axis='both', which='major', labelsize=14, colors='black')
    plt.tight_layout()
    
    if save_path:
        save_path = Path(save_path)
        plt.savefig(save_path.with_suffix('.png'), dpi=300, bbox_inches='tight')    
    plt.show()
    
    return fig, ax, selected_indices

### Running the code for each dataset
____

In [ ]:
for dataset in list_datasets:
    df = pd.DataFrame()
    print(f'Processing dataset: {dataset}' '\n') 
    folder_with_files, plot_name, dataframe_prefix = dataset_selection(dataset,control_spots_mode, downsample)
    array_int_all_days, total_number_of_cells, total_number_of_spots = load_tracking_data(folder_with_files, selected_field=selected_field,min_percentage_data_in_trajectory=min_percentage_data_in_trajectory,dataframe_prefix=dataframe_prefix,min_snr=min_snr)
    plot_data_matrix(array_int_all_days, results_folder, plot_name)
    fig, ax, selected_indices = plot_trajectory_timecourses(
                                array_int_all_days,
                                num_trajectories_plot=2,
                                figsize=(12, 3),
                                alpha=0.7,
                                linewidth=1.5,
                                time_interval_seconds=step_size_in_sec,
                                #title="High Quality Trajectories",
                                save_path= results_folder / f'trajectories_{plot_name}' 
                            )
    if downsample:
        primary_data = mi.Utilities().downsample_array(array_int_all_days, factor=downsampling_factor, method='average')
        step_size_in_sec_downsampled = step_size_in_sec * downsampling_factor
    else:
        primary_data = array_int_all_days
        step_size_in_sec_downsampled = step_size_in_sec
    # Temporarily disable parallel processing to avoid module import issues
    with joblib.parallel_backend('threading', n_jobs=1):
        # calculate the ACF
        mean_correlation, std_correlation, lags, correlations_array, dwell_time = mi.Correlation(primary_data=primary_data,
                                                                                            max_lag=None, 
                                                                                            nan_handling= 'forward_fill',  #forward_fill, 'ignore'
                                                                                            shift_data=True,
                                                                                            return_full=False,
                                                                                            time_interval_between_frames_in_seconds=step_size_in_sec_downsampled,
                                                                                            use_bootstrap=True,
                                                                                            show_plot=False,
                                                                                            start_lag=start_lag,
                                                                                            fit_type='exponential',
                                                                                            de_correlation_threshold=0.005,
                                                                                            correct_baseline=True,
                                                                                            use_linear_projection_for_lag_0=True,
                                                                                            save_plots=False,
                                                                                            use_global_mean= False,
                                                                                            remove_outliers = remove_outliers,
                                                                                            MAD_THRESHOLD_FACTOR = MAD_THRESHOLD_FACTOR,
                                                                                            plot_individual_trajectories = False,
                                                                                            y_axes_min_max_list_values = None, #y_axes_min_max_list_values,
                                                                                            x_axes_min_max_list_values=x_axes_min_max_list_values,
                                                                                            multi_tau=multi_tau,
                                                                                            multi_tau_raw_points = multi_tau_raw_points,
                                                                                            multi_tau_bins_per_stage = multi_tau_bins_per_stage,
                                                                                            plot_title=None).run()
    # save all in a single dataframe the rows are each lag, the columns are the mean_correlation, std_correlation, correlations_array, dwell_time
    df = pd.DataFrame(data={'lags': lags, 'mean_correlation': mean_correlation, 'std_correlation': std_correlation})
    df_name ='df_ACF_'+plot_name
    if control_spots_mode:
        df_name = df_name+'_control_spots'
    df.to_csv(results_folder.joinpath(df_name+'.csv'), index=False)
    number_of_trajectories_final = np.shape(correlations_array)[0]
    # write a text file with the results
    file = open(results_folder.joinpath('report_cells_trajectories_ACF_'+plot_name+'.txt'), 'w')
    file.write(f'Total number of cells: {total_number_of_cells}\n')
    file.write(f'Total number of spots: {total_number_of_spots}\n')
    file.write(f'Number of trajectories after filtering based on quality: {number_of_trajectories_final}\n')
    file.close()
    print(f'Number of trajectories: {np.shape(correlations_array)[0]}')
    print(f'Number of cells: {total_number_of_cells}')
    print('-----------------------------------------------------\n')
    plot_correlation_comparison(mean_correlation, std_correlation, lags, plot_name=plot_name, start_lag=0, show_legend = False, y_axes_min_max_list_values = y_axes_min_max_list_values,x_axes_min_max_list_values=x_axes_min_max_list_values)